In [0]:
# widgets are used for parameters  at jobs clusters/workflow
dbutils.widgets.text("myname", "Gopal")
name = dbutils.widgets.get("myname")  # read data from widget
print ("name", name)

name Gopalakrishnan


In [0]:
# remove widget by name
# dbutils.widgets.remove("myname")
# rmeove all widgets
dbutils.widgets.removeAll()
     

In [0]:
# Convert CSV files into PArquet format
# Delta Lake: Move data from Bronze to Silver
# Row based CSV foramt to Column based Parquet


dbutils.widgets.text("moviesPath", "/Volumes/workspace/default/bronze/movies/")
# target must be silver
dbutils.widgets.text("moviesTargetPath", "/Volumes/workspace/default/silver/movies")

In [0]:

MOVIES_PATH = dbutils.widgets.get("moviesPath") 
print (MOVIES_PATH)

/Volumes/workspace/default/bronze/movies/


In [0]:
# how to create schema programatically instead of using inferSchema
from pyspark.sql.types import StructType, LongType, StringType, IntegerType, DoubleType

In [0]:
# True is nullable, False is non nullable
# give your own column name, datatypes
movieSchema = StructType()\
                .add("movieId", IntegerType(), True)\
                .add("title", StringType(), True)\
                .add("genres", StringType(), True)
     

In [0]:
# movieDf with schema we have, to avoid additional job creation, scan data overload
# since we provide schema, header = true, is to skip the first line in the csv
movieDf = (spark
            .read
            .option("header", True)
            .schema(movieSchema) # now , we don't run a job for schema
            .csv(MOVIES_PATH)
            )
movieDf.printSchema()
movieDf.show(5)
# make a note of number of jobs its create, compare with previous shell

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


In [0]:
# movie data, convert to parquet as is
# silver container
# movies directory

# move to silver container
# Put 
# Put MOVIE_TARGET_PATH as part of the widget

MOVIE_TARGET_PATH = dbutils.widgets.get("moviesTargetPath")
# overwrite - delete old data
# write is a batch write
movieDf.write.mode("overwrite").parquet(MOVIE_TARGET_PATH)
     

In [0]:
# useful to pass result of one notebook to another notebook in jobs compute
dbutils.jobs.taskValues.set(key = "movies_silver_path", value = MOVIE_TARGET_PATH)